### Aplicações - Estudo Dirigido AV1

#### Exercício 2.13: 

a) Implemente $erm(X,y,G,loss)$, que recebe uma lista de funções $G$ e uma perda $loss(y, y_{hat})$ e devolve a função de menor risco empírico. 

b) Use-a com $G = {g_b(x) = bx: b \in {0,0,1,0,2,...,5}}$ e os dados do Exercício teórico 2.9, com $l_2$ e $l_1$. Compare com $\hat{b}$ calculado a mão. 


In [3]:
def loss_l1(y,y_hat):
    #Perda l1 (absoluta)
    return abs(y - y_hat)

def loss_l2(y,y_hat):
    #Perda l2 (quadratica)
    return (y-y_hat)**2

def erm(X,y,G,loss):
    #Funcao auxiliar para calcular a media das perdas de uma funcao g
    def calcular_risco(g): 
        erros=[loss(yi,g(xi)) for xi,yi in zip(X,y)]
        return sum(erros)/len(erros)
     
    return min(G, key = calcular_risco)


# Dados do Exercício 2.9
X = [1, 2, 3, 4]
y = [3, 4, 8, 10]

# Construção do grid de b: [0.0, 0.1, 0.2, ..., 5.0]
grid_b = [i / 10.0 for i in range(51)]

# Família G de funções gb(x) = b * x
G = [lambda x, b=b: b * x for b in grid_b]

# Minimização com perda L2
melhor_g_l2 = erm(X, y, G, loss_l2)
b_chapeu_l2 = melhor_g_l2(1)

# Minimização com perda L1
melhor_g_l1 = erm(X, y, G, loss_l1)
b_chapeu_l1 = melhor_g_l1(1)

print(f"(L2) encontrado via ERM: {b_chapeu_l2:.2f}")
print(f"(L1) encontrado via ERM: {b_chapeu_l1:.2f}")

(L2) encontrado via ERM: 2.50
(L1) encontrado via ERM: 2.50


Mesmo valor encontrado com o cálculo à mão :)

#### Exercício 2.14 ( KNN de regressão)

Implemente, sem bibliotecas além de $math$ e $random$, $ knn.reg(X,y,x0,k) $, que devolve a média dos $yi$ dos $k$ pontos mais próximos de $x0$. Gere $N = 100$ pontos de $y = sin(2\pi x) + \epsilon$ em [0,1] e avalie $\hat{f}$ em uma grande fina para $k \in {1,5,20}$. Descreva o que muda. 

In [5]:
import math
import random

def knn_reg(X,y,x0,k):
    #Calcula a distancia de xo para todos os pontos xi do conjunto de dados X
    dist = [abs(xi-x0) for xi in X]
    
    in_pares = list(zip(dist,y)) #Juntar xi por seu respectivo yi
    #Ordena de acordo com as menores distancias xo dos xs
    in_pares_ordenados = sorted(in_pares, key = lambda par: par[0]) 
    
    k_vizinhos = in_pares_ordenados[:k]
    
    y_vizinhos = [par[1] for par in k_vizinhos]
    
    #Retorna a média simples desses k y_vizinhos
    return sum(y_vizinhos)/k 


# Fixa a semente para garantir que os números aleatórios sejam reproduzíveis
random.seed(42)

N = 100
sigma = 0.2

# Gera 100 valores aleatórios de X uniformemente distribuídos em [0, 1]
X_train = [random.uniform(0, 1) for _ in range(N)]

# Gera Y_i = sin(2 * pi * x_i) + ruido_gaussiano
y_train = [
    math.sin(2 * math.pi * x) + random.gauss(0, sigma) 
    for x in X_train
]

# Cria uma grade fina com 200 pontos entre 0 e 1
num_pontos_grid = 200
X_grid = [i / (num_pontos_grid - 1) for i in range(num_pontos_grid)]

# Dicionário para armazenar as previsões de cada k
predicoes = {}

for k in [1, 5, 20]:
    # Para cada x0 na grade, calcula a estimativa usando knn_reg
    predicoes[k] = [knn_reg(X_train, y_train, x0, k) for x0 in X_grid]
    
# Exibe a comparação das predições em 5 pontos estratégicos da grade
indices_amostra = [0, 50, 100, 150, 199]

print("x0     | Real (seno) | k=1    | k=5    | k=20")
print("-" * 45)
for idx in indices_amostra:
    x0 = X_grid[idx]
    y_real = math.sin(2 * math.pi * x0)
    p1 = predicoes[1][idx]
    p5 = predicoes[5][idx]
    p20 = predicoes[20][idx]
    print(f"{x0:.2f}   | {y_real:+.3f}      | {p1:+.3f} | {p5:+.3f} | {p20:+.3f}")

x0     | Real (seno) | k=1    | k=5    | k=20
---------------------------------------------
0.00   | +0.000      | -0.032 | +0.203 | +0.500
0.25   | +1.000      | +1.103 | +0.894 | +0.976
0.50   | -0.016      | -0.144 | -0.177 | -0.097
0.75   | -1.000      | -1.027 | -0.946 | -0.825
1.00   | -0.000      | +0.082 | -0.024 | -0.487


Para k=1 (alta variância, baixo viés) ocorre overfitting, para k=5 (equilíbrio) e para k=20 (baixa variância, alto viés) ocorre um underfitting.

#### Exercício 2.25

a) Reescreva $particao.dados$ do capítulo para matrizes Numpy, devolvendo índices de treino e teste.

b) Implemente $k.fold(X,y,k,fit,predict,loss)$, em que $fit(X,y)# devolve um modelo, $predict(modelo,X)$ devolve previsões e $loss$ é a perda. A função devolve o erro médio de validação cruzada. 

c) Use-a para escolher o $k$ do $kNN$ do $Exercício 2.14$ em dados sintéticos. Compare com o k que minimiza o erro de treino.  

In [6]:
import numpy as np

def particao_dados(N, prop_treino=0.8, seed=42):
    if seed is not None: 
        np.random.seed(seed)
    
    # Gera uma lista embaralhada de índices de 0 a N-1
    indices = np.random.permutation(N)
    
    # Define o ponto de corte
    n_treino = int(N * prop_treino)
    
    idx_treino = indices[:n_treino]
    idx_teste = indices[n_treino:]
    
    return idx_treino, idx_teste

def k_fold(X,y,k,fit,predict,loss):
    
    N=len(y)
    
    indices = np.random.permutation(N)
    folds = np.array_split(indices,k)
    
    erros_folds = []
    for i in range(k):
        idx_teste = folds[i]
        idx_treino = np.concatenate([folds[j] for j in range(k) if j != i])
        
        X_tr, y_tr = X[idx_treino], y[idx_treino]
        X_te, y_te = X[idx_teste], y[idx_teste]
        
        modelo = fit(X_tr, y_tr)
        y_pred = predict(modelo, X_te)
        
        erros_folds.append(loss(y_te, y_pred))
        
    return np.mean(erros_folds)
     